# Day 3 · Vision 모의 실기

12문항 / 100점. ResNet18·ViT·U-Net·DETR와 Intro PDF의 이미지 전처리 코드를 바탕으로 구성했습니다.
권장 첫 풀이 시간은 150~210분의 가안입니다. PDF의 평가 축은 레이어 / 학습 연결 / 데이터 변환 / 평가 지표입니다.
**코드의 실제 축 순서는 PyTorch 기준 `(B,C,H,W)`입니다.** 강의 그림의 B,C,W,H 표기와 구분하세요.
정사각형에서는 숫자가 같지만 직사각형은 다릅니다. PIL의 size만 `(W,H)`입니다.

**연습용 모의시험입니다. 실제 기출·공식 배점·공식 제한 시간이 아닙니다.**
근거: [인증시험 출제 포인트 PDF](../../AI인증시험_출제포인트.pdf) 1쪽과 문제별 교안 셀.
셀 번호는 Markdown을 포함하여 맨 위부터 1로 센 번호입니다.

## 푸는 방법

1. 이 파일을 복사하여 `-submission.ipynb` 이름으로 저장합니다. 기존 실습 답안은 수정하지 않습니다.
2. `env_aias_test` 커널을 선택하고 아래 **공통 준비 셀**부터 실행합니다.
3. 문제의 `????`를 코드로 바꾸거나 `raise NotImplementedError` 자리에 필요한 몇 줄을 씁니다.
`????`는 의도적인 빈칸이므로 수정 전 해당 답안 셀은 SyntaxError가 납니다.
4. 답안 셀 → 바로 아래 확인 셀 순서로 실행합니다. 확인 셀은 수정하지 않습니다.
각 문제는 공통 준비 셀만 있으면 독립적으로 풀 수 있습니다.
5. 이유·shape 답변도 작성하고, 끝나면 **Restart Kernel → Run All**로 재검사합니다.
못 푼 문제는 번호를 기록하고 다음 문제로 이동할 수 있습니다.

데이터·작은 모델·인자는 제공됩니다. 인터넷 접속, 파일 다운로드, 설치, GPU, 장시간 학습은 필요하지 않습니다.
예제 입력의 숫자나 배치 크기를 답안에 고정하지 말고 전달된 인자를 사용하세요.
확인 셀 통과는 공개 사례의 동작 확인이며 전체 점수·개념 숙달을 보장하지 않습니다.
문제당 점수는 코드 요구 충족 80%, 이유·shape 설명 20%로 채점할 연습용 기준입니다.
정답 코드는 포함하지 않았습니다. 막힌 문제 번호를 알려주면 힌트부터 제공합니다.


## 공통 준비 · 제공 코드 / 수정하지 마세요


In [ ]:
import math
import copy
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
torch.manual_seed(123)
torch.set_num_threads(2)
device = torch.device("cpu")
print("PyTorch:", torch.__version__, "| device:", device)

class TinySequenceModel(nn.Module):
    # 문제 독립 실행용 작은 대체 모델. 실제 GPT 성능을 재현하지 않습니다.
    def __init__(self, vocab=16, dim=8, out=16):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim)
        self.head = nn.Linear(dim, out)
    def forward(self, ids):
        return self.head(self.emb(ids))

from PIL import Image
from torchvision import transforms as T, models

# ViT 교안의 einops Rearrange를 설치 없이 실행하기 위한 제공 대역입니다.
# 교안과 같은 patch 순서: (grid_h, grid_w), 내부 순서: (patch_h, patch_w, channel).
class ProvidedPatches(nn.Module):
    def __init__(self, patch_size): super().__init__(); self.p=patch_size
    def forward(self,x):
        b,c,h,w=x.shape; p=self.p
        assert h%p == 0 and w%p == 0
        return x.reshape(b,c,h//p,p,w//p,p).permute(0,2,4,3,5,1).reshape(b,(h//p)*(w//p),p*p*c)


## VISION01 · 이미지 변환·정규화·배치 축 — 8점

**출처:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 셀 6

**평가 축: torchvision.transforms·데이터 구성.** 추가 근거: [Intro PDF](../../vision/01_Intro.pdf) 17쪽.
PIL 이미지와 mean/std를 받아 RGB 변환→ToTensor→Normalize→배치 축 추가 순서로 처리하세요.
`train=True`일 때만 ToTensor 전에 RandomCrop(32,padding=4), RandomHorizontalFlip()을 넣습니다.
평가 시 크기는 보존합니다. 제공 입력은 uint8 PIL 이미지입니다. 반환 `(1,3,H,W)` float32.

**짧은 설명 문제:** ToTensor와 Normalize는 픽셀 값에 각각 어떤 변화를 주나요? 평가에 랜덤 증강을 넣지 않는 이유도 적으세요.


In [ ]:
def image_to_batch(image, mean, std, train=False):
    image=image.convert(????)
    ops=[]
    if train:
        ops += [T.RandomCrop(32,padding=4), T.RandomHorizontalFlip()]
    ops += [T.????(), T.????(mean,std)]
    transform=T.????(ops)
    return transform(image).????(????)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
image=Image.new('RGB',(4,2),(255,128,0))
x=image_to_batch(image,(.5,.5,.5),(.5,.5,.5))
assert x.shape == (1,3,2,4) and x.dtype == torch.float32
torch.testing.assert_close(x[0,:,0,0],torch.tensor([1.,(128/255-.5)/.5,-1.]))
gray=Image.new('L',(4,2),255)
assert image_to_batch(gray,(0,0,0),(1,1,1)).shape == (1,3,2,4)
assert image_to_batch(Image.new('RGB',(32,32)),(0,0,0),(1,1,1),True).shape == (1,3,32,32)
print("VISION01 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION02 · 분류용 DataLoader 구성 — 6점

**출처:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 셀 6 · [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 셀 6

**평가 축: 학습·평가 목적에 맞는 배치.** images `(N,3,32,32)` float와 labels `(N,)` long이 제공됩니다.
TensorDataset으로 묶고, 학습일 때만 shuffle=True로 설정하세요. drop_last=False, num_workers=0입니다.
DataLoader를 반환하세요. 다운로드 및 실제 CIFAR 파일 준비는 문제에 없습니다.

**짧은 설명 문제:** 데이터 5개, batch_size=2에서 배치별 이미지·라벨 shape를 적으세요.


In [ ]:
def vision_loader(images, labels, batch_size, train):
    dataset=TensorDataset(????,????)
    return DataLoader(dataset,batch_size=????,shuffle=????,drop_last=????,num_workers=0)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
images=torch.randn(5,3,32,32)
labels=torch.tensor([0,1,2,3,4])
loader=vision_loader(images,labels,2,False)
batches=list(loader)
assert [x.shape[0] for x,y in batches] == [2,2,1]
assert torch.cat([y for x,y in batches]).tolist() == [0,1,2,3,4]
assert batches[0][1].dtype == torch.long
from torch.utils.data import RandomSampler, SequentialSampler
assert isinstance(loader.sampler,SequentialSampler)
assert isinstance(vision_loader(images,labels,2,True).sampler,RandomSampler)
print("VISION02 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION03 · CIFAR용 ResNet18 모델 구성 — 10점

**출처:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 셀 9

**평가 축: 주요 레이어·모델 구성.** torchvision ResNet18을 CIFAR용으로 수정하세요.
conv1은 RGB→64, kernel=3,stride=1,padding=1,bias=False, maxpool은 항등 연산, fc는 num_classes 출력입니다.
`weights=None`은 제공되며 다운로드하지 않습니다. 입력 `(B,3,32,32)` → `(B,num_classes)` logits.

**짧은 설명 문제:** layer4→avgpool→flatten→fc의 shape와, residual의 더하기가 채널 수를 두 배로 만들지 않는 이유를 적으세요.


In [ ]:
def build_cifar_resnet(num_classes):
    model=models.resnet18(weights=None)
    model.conv1=nn.Conv2d(????,????,kernel_size=????,stride=????,padding=????,bias=False)
    model.maxpool=nn.????()
    model.fc=nn.Linear(????,????)
    return model


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
m=build_cifar_resnet(10).eval()
assert m.conv1.in_channels == 3 and m.conv1.out_channels == 64
assert m.conv1.kernel_size == (3,3) and m.conv1.stride == (1,1) and m.conv1.padding == (1,1)
assert isinstance(m.maxpool,nn.Identity) and m.fc.in_features == 512
shapes={}
handles=[]
def remember(name):
    def hook(module,args,out): shapes[name]=tuple(out.shape)
    return hook
for name in ['layer1','layer2','layer3','layer4','avgpool']:
    handles.append(getattr(m,name).register_forward_hook(remember(name)))
with torch.no_grad(): result=m(torch.randn(2,3,32,32))
for h in handles: h.remove()
assert result.shape == (2,10)
assert shapes == {'layer1':(2,64,32,32),'layer2':(2,128,16,16),'layer3':(2,256,8,8),'layer4':(2,512,4,4),'avgpool':(2,512,1,1)}
assert build_cifar_resnet(3).fc.out_features == 3
print(shapes)
print("VISION03 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION04 · 분류 학습·평가 루프 — 12점

**출처:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 셀 11 · [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 셀 19~20

**평가 축: 예측값·정답을 학습으로 연결.** 한 epoch 함수를 완성하세요. optimizer가 있으면 학습, None이면 평가입니다.
학습일 때만 gradient 초기화·역전파·갱신을 합니다. 평가에서는 eval과 gradient 기록 금지가 모두 필요합니다.
criterion은 배치 평균 손실, loader는 비어 있지 않습니다. 반환은 `(샘플별 평균 loss, 전체 accuracy)`의 float 두 개입니다.
교안 AMP는 CPU에서 핵심 순서를 확인하도록 제외한 변형입니다. 정확도는 해당 forward의 logits 기준입니다.

**짧은 설명 문제:** CrossEntropyLoss 앞에 softmax를 따로 넣지 않는 이유, 배치 loss에 bs를 곱하는 이유를 적으세요.


In [ ]:
def classification_epoch(model, loader, criterion, device, optimizer=None):
    training=optimizer is not None
    model.train(????)
    total_loss,correct,count=0.0,0,0
    with torch.set_grad_enabled(????):
        for images,labels in loader:
            images,labels=images.to(device),labels.to(device)
            if training: optimizer.????()
            logits=model(images)
            loss=criterion(????,????)
            if training:
                loss.????()
                optimizer.????()
            bs=labels.size(0)
            total_loss += loss.item() * ????
            correct += (logits.argmax(dim=????)==labels).sum().item()
            count += bs
    return total_loss/count,correct/count


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
class ModeLinear(nn.Linear):
    def forward(self,x):
        assert torch.is_grad_enabled() == self.training
        return super().forward(x)
model=ModeLinear(2,2)
x=torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
y=torch.tensor([0,1,1])
loader=DataLoader(TensorDataset(x,y),batch_size=2)
criterion=nn.CrossEntropyLoss()
model.eval()
with torch.no_grad():
    z=model(x); expected=criterion(z,y).item(); expected_acc=(z.argmax(1)==y).float().mean().item()
before=copy.deepcopy(model.state_dict())
loss,acc=classification_epoch(model,loader,criterion,device)
assert abs(loss-expected)<1e-6 and abs(acc-expected_acc)<1e-6
assert not model.training
for name,tensor in model.state_dict().items(): torch.testing.assert_close(tensor,before[name])
opt=torch.optim.SGD(model.parameters(),lr=.1)
classification_epoch(model,loader,criterion,device,opt)
assert model.training and not torch.equal(model.weight,before['weight'])
print("VISION04 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION05 · Confusion matrix·클래스별 정확도 — 8점

**출처:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 셀 17 · [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 셀 27

**평가 축: 평가 Metric.** `preds`, `labels`는 `(N,)` long입니다. 혼동행렬은 **행=실제 라벨, 열=예측**으로 누적하세요.
반환: `(C,C)` int64 혼동행렬, `(C,)` float 클래스별 정답 비율, Python float 전체 정확도.
실제 샘플이 없는 클래스의 비율은 0으로 둡니다. N은 1 이상입니다.

**짧은 설명 문제:** 클래스별 정답 비율의 분모가 행 합인 이유와 전체 정확도와의 차이를 적으세요.


In [ ]:
def classification_metrics(preds, labels, num_classes):
    cm=torch.zeros(num_classes,num_classes,dtype=torch.int64)
    for truth,pred in zip(labels,preds):
        cm[int(????),int(????)] += 1
    per_class=cm.diagonal().float()/cm.sum(dim=????).clamp_min(1)
    accuracy=????
    return cm,per_class,accuracy


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
pred=torch.tensor([0,0,1,1,2]); truth=torch.tensor([0,1,1,2,2])
cm,pc,acc=classification_metrics(pred,truth,4)
assert cm.tolist()==[[1,0,0,0],[1,1,0,0],[0,1,1,0],[0,0,0,0]]
torch.testing.assert_close(pc,torch.tensor([1.,.5,.5,0.]))
assert isinstance(acc,float) and abs(acc-.6)<1e-7
print("VISION05 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION06 · ViT 패치 임베딩 — 8점

**출처:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 셀 11

**평가 축: 레이어·Tensor 흐름.** P×P 패치의 수와 한 패치의 펼친 길이를 계산하여 Linear를 구성하세요.
`ProvidedPatches`는 교안의 einops Rearrange와 같은 순서를 구현한 제공 코드입니다. 입력 `(B,C,H,W)`에서 `(B,N,P×P×C)`를 만듭니다.
H,W는 P로 나누어떨어집니다. 출력은 `(B,N,dim)`입니다. 이미지 크기가 바뀌어도 공식으로 계산하세요.

**짧은 설명 문제:** 32×32 RGB 이미지를 4×4 패치로 나눌 때 64와 48이 각각 무엇인지 적으세요.


In [ ]:
class ExamPatchEmbedding(nn.Module):
    def __init__(self, height, width, channels, patch_size, dim):
        super().__init__()
        self.num_patches=????
        patch_dim=????
        self.net=nn.Sequential(ProvidedPatches(patch_size),nn.Linear(patch_dim,dim))
    def forward(self,x): return self.net(x)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
m=ExamPatchEmbedding(32,32,3,4,16)
assert m.num_patches==64 and m.net[1].in_features==48
assert m(torch.randn(2,3,32,32)).shape==(2,64,16)
m2=ExamPatchEmbedding(8,12,1,4,7)
assert m2.num_patches==6 and m2(torch.randn(1,1,8,12)).shape==(1,6,7)
print("VISION06 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION07 · ViT Self-Attention·헤드 결합 — 8점

**출처:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 셀 9 / Attention

**평가 축: 레이어 동작.** QKV 통합 투영과 여러 헤드의 attention을 구현하세요. 이 Vision encoder에는 causal mask를 적용하지 않습니다.
`dim=heads*head_dim`으로 제한한 축소 문제입니다. 입력 `(B,N,dim)`, 출력 같은 shape, weights `(B,H,N,N)`.
교안 einsum/rearrange는 @/reshape/transpose로 바꿔도 같은 연산입니다. dropout은 생략했습니다.

**짧은 설명 문제:** GPT causal attention과 달리 이미지 패치의 뒤쪽 위치도 볼 수 있는 이유를 적으세요.


In [ ]:
class ExamVisionAttention(nn.Module):
    def __init__(self,dim,heads):
        super().__init__()
        assert dim%heads==0
        self.h=heads
        self.to_qkv=nn.Linear(dim,????,bias=False)
        self.to_out=nn.Linear(dim,dim)
    def forward(self,x):
        b,n,d=x.shape
        q,k,v=self.to_qkv(x).chunk(????,dim=-1)
        q,k,v=[z.reshape(b,n,self.h,d//self.h).transpose(1,2) for z in (q,k,v)]
        scores=(q @ k.transpose(-2,-1))/math.sqrt(????)
        weights=torch.softmax(scores,dim=????)
        out=(weights @ ????).transpose(1,2).reshape(b,n,d)
        return self.to_out(out),weights


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
m=ExamVisionAttention(8,2)
with torch.no_grad():
    m.to_qkv.weight[:16].zero_()  # Q,K=0이면 모든 위치의 점수가 같습니다.
x=torch.randn(2,5,8)
y,w=m(x)
assert y.shape==(2,5,8) and w.shape==(2,2,5,5)
torch.testing.assert_close(w,torch.full_like(w,.2))
vals=F.linear(x,m.to_qkv.weight[16:])
torch.testing.assert_close(y,m.to_out(vals.mean(1,keepdim=True).expand(-1,5,-1)))
numeric=ExamVisionAttention(8,2)
z=torch.randn(1,3,8)
q,k,v=[t.reshape(1,3,2,4).transpose(1,2) for t in numeric.to_qkv(z).chunk(3,dim=-1)]
reference=F.scaled_dot_product_attention(q,k,v).transpose(1,2).reshape(1,3,8)
torch.testing.assert_close(numeric(z)[0],numeric.to_out(reference))
print("VISION07 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION08 · CLS·위치 임베딩·분류 head — 8점

**출처:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 셀 11 / forward

**평가 축: 전체 모델 흐름.** 이미 만들어진 patch tokens `(B,N,D)`와 encoder를 받습니다.
학습 가능한 CLS를 배치 수만큼 복제해 맨 앞에 붙이고 위치 임베딩을 더합니다. encoder 후 첫 토큰을 head에 전달하세요.
출력 `(B,num_classes)` raw logits. 이 문제는 cls pooling만 구현합니다. N은 생성자 num_patches와 같습니다.

**짧은 설명 문제:** 패치가 64개일 때 attention map이 64×64가 아니라 65×65가 되는 이유를 적으세요.


In [ ]:
class ExamViTHead(nn.Module):
    def __init__(self,num_patches,dim,num_classes,encoder):
        super().__init__()
        self.cls=nn.Parameter(torch.randn(1,1,dim))
        self.pos=nn.Parameter(torch.randn(1,num_patches+1,dim))
        self.encoder=encoder
        self.head=nn.Sequential(nn.LayerNorm(dim),nn.Linear(dim,num_classes))
    def forward(self,tokens):
        b,n,d=tokens.shape
        cls=self.cls.expand(????,-1,-1)
        x=torch.cat((cls,tokens),dim=????)
        x=x+self.pos[:,:????]
        x=self.encoder(x)
        return self.head(x[:,????,:])


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
class MixEncoder(nn.Module):
    def forward(self,x): return x + x.mean(dim=1,keepdim=True)
m=ExamViTHead(4,6,10,MixEncoder())
tokens=torch.randn(2,4,6)
y=m(tokens)
assert y.shape==(2,10)
seq=torch.cat((m.cls.expand(2,-1,-1),tokens),1)+m.pos
torch.testing.assert_close(y,m.head(m.encoder(seq)[:,0]))
y.square().sum().backward()
assert m.cls.grad is not None and m.pos.grad is not None
print("VISION08 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION09 · U-Net DoubleConv·Down — 8점

**출처:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 셀 7 / DoubleConv, Down

**평가 축: 레이어 구성.** `(Conv3×3→ReLU)`를 두 번 적용하는 DoubleConv를 완성하세요.
두 Conv 모두 stride=1,padding=1,bias=False. 중간 채널=out_channels입니다.
Down은 MaxPool2d(2) 뒤 DoubleConv입니다. 입력 공간 크기는 짝수입니다.

**짧은 설명 문제:** DoubleConv와 MaxPool 각각이 공간 크기·채널 수에 주는 영향을 설명하세요.


In [ ]:
class ExamDoubleConv(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(????,????,kernel_size=3,padding=????,bias=False),nn.ReLU(),
            nn.Conv2d(????,????,kernel_size=3,padding=1,bias=False),nn.ReLU())
    def forward(self,x): return self.net(x)

class ExamDown(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        self.pool=nn.MaxPool2d(????)
        self.conv=ExamDoubleConv(in_channels,out_channels)
    def forward(self,x): return self.????(self.????(x))


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
m=ExamDoubleConv(3,8)
x=torch.randn(2,3,16,24)
assert m(x).shape==(2,8,16,24)
down=ExamDown(3,8)
assert down(x).shape==(2,8,8,12)
assert sum(isinstance(layer,nn.Conv2d) for layer in m.modules())==2
assert m.net[0].padding==(1,1) and m.net[0].kernel_size==(3,3)
print("VISION09 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION10 · U-Net Up·skip 채널 결합 — 8점

**출처:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 셀 7 / Up

**평가 축: 레이어 연결·shape.** 교안의 bilinear Up 경로를 완성하세요. up과 conv는 생성자에서 제공합니다.
decoder를 2배 업샘플하고 skip과 높이·너비가 같도록 대칭 padding을 적용한 다음, `[skip,decoder]` 순서로 채널 결합하세요.
padding 계산은 제공됩니다. conv의 입력 채널은 두 입력의 채널 합입니다. decoder 업샘플 크기는 skip 이하라고 가정합니다.

**짧은 설명 문제:** ResNet의 residual 더하기와 U-Net skip의 concat은 채널 수가 어떻게 다른가요?


In [ ]:
class ExamUp(nn.Module):
    def __init__(self,conv):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2,mode="bilinear",align_corners=True)
        self.conv=conv
    def forward(self,decoder,skip):
        decoder=self.????(decoder)
        dy=skip.size(2)-decoder.size(2)
        dx=skip.size(3)-decoder.size(3)
        decoder=F.pad(decoder,[dx//2,dx-dx//2,dy//2,dy-dy//2])
        merged=torch.????([skip,decoder],dim=????)
        return self.????(merged)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
decoder=torch.ones(2,4,3,4)*2
skip=torch.ones(2,3,7,9)
raw=ExamUp(nn.Identity())(decoder,skip)
assert raw.shape==(2,7,7,9)
torch.testing.assert_close(raw[:,:3],skip)
assert raw[:,3:,0,0].eq(2).all() and raw[:,3:,-1,-1].eq(0).all()
out=ExamUp(nn.Conv2d(7,2,3,padding=1))(decoder,skip)
assert out.shape==(2,2,7,9)
print("VISION10 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION11 · 이진 segmentation 손실·마스크 — 8점

**출처:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 셀 9, 11, 15

**평가 축: 예측·정답→학습·추론.** logits와 masks는 `(B,1,H,W)` float32이며 mask 값은 0 또는 1입니다.
손실은 BCEWithLogitsLoss를 사용하고, 추론 마스크는 sigmoid 확률이 threshold보다 **클 때** True로 만듭니다.
반환: scalar loss, 같은 shape의 bool 예측. 분류처럼 argmax를 사용하지 마세요.

**짧은 설명 문제:** 이미지 분류의 (B,) 정답과 달리 segmentation 정답에 H,W가 남는 이유를 적으세요.


In [ ]:
def segmentation_loss_and_mask(logits,masks,threshold=.5):
    criterion=nn.????()
    loss=criterion(????,????)
    pred=torch.????(logits) > threshold
    return loss,pred


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
logits=torch.tensor([[[[-2.,2.],[0.,-1.]]]],requires_grad=True)
masks=torch.tensor([[[[0.,1.],[0.,0.]]]])
loss,pred=segmentation_loss_and_mask(logits,masks)
assert loss.ndim==0 and pred.dtype==torch.bool
assert pred.tolist()==[[[[False,True],[False,False]]]]
torch.testing.assert_close(loss,F.binary_cross_entropy_with_logits(logits,masks))
assert not segmentation_loss_and_mask(logits,masks,.99)[1].any()
loss.backward(); assert logits.grad is not None
print("VISION11 공개 확인 통과 — 설명 답변도 작성하세요.")


## VISION12 · DETR confidence·박스 좌표 복원 — 8점

**출처:** [03_DETR.ipynb](../../vision/03_DETR.ipynb) 셀 8, 18

**평가 축: 모델 출력 활용·데이터 변환.** outputs에는 pred_logits `(1,Q,C+1)`, pred_boxes `(1,Q,4)`가 있습니다.
마지막 클래스는 no-object입니다. **전체 C+1 클래스에 softmax한 뒤** no-object 열을 제거하세요.
객체 클래스 최대 확률이 threshold보다 큰 쿼리만 남기고, 정규화된 `(cx,cy,w,h)`를 픽셀 `(xmin,ymin,xmax,ymax)`로 바꾸세요.
size는 PIL 기준 `(너비,높이)`. 반환은 필터된 클래스 확률 `(K,C)`와 박스 `(K,4)`입니다. 모델은 제공 출력으로 대체합니다.

**짧은 설명 문제:** no-object 열을 softmax 전에 제거하면 어떤 오탐이 생길 수 있나요? width,height의 곱셈 순서도 설명하세요.


In [ ]:
def detr_postprocess(outputs,size,threshold=.9):
    prob=outputs['pred_logits'].softmax(dim=????)[0,:,????]
    keep=prob.max(dim=-1).values > ????
    boxes=outputs['pred_boxes'][0,????]
    cx,cy,w,h=boxes.unbind(1)
    xyxy=torch.stack((????,????,????,????),dim=1)
    width,height=size
    scale=boxes.new_tensor([????,????,????,????])
    return prob[keep],xyxy*scale


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
outputs={'pred_logits':torch.tensor([[[8.,0.,-3.],[0.,0.,8.],[-3.,8.,0.]]]),
         'pred_boxes':torch.tensor([[[.5,.5,.4,.2],[.5,.5,1.,1.],[.25,.25,.2,.2]]])}
prob,boxes=detr_postprocess(outputs,(200,100),.9)
assert prob.shape==(2,2) and boxes.shape==(2,4)
assert prob.argmax(-1).tolist()==[0,1]
torch.testing.assert_close(boxes,torch.tensor([[60.,40.,140.,60.],[30.,15.,70.,35.]]))
# no-object를 먼저 제거한 뒤 softmax하면 이 쿼리를 잘못 남깁니다.
tricky={'pred_logits':torch.tensor([[[8.,0.,10.]]]),'pred_boxes':torch.tensor([[[.5,.5,.2,.2]]])}
assert detr_postprocess(tricky,(200,100),.9)[0].shape==(0,2)
assert detr_postprocess(outputs,(200,100),1.0)[1].shape==(0,4)
print("VISION12 공개 확인 통과 — 설명 답변도 작성하세요.")


## 제출 전 확인

- 모든 답안 셀의 빈칸과 NotImplementedError를 해결했나요?
- 새 커널에서 공통 준비→답안→확인 셀 전체가 실행되나요?
- 이유·shape 답변과 도움 사용 여부를 적었나요?
- 틀린 문제 번호와 오류 메시지를 아래에 남겨 주세요. 다음 복습에서는 그 구간만 인자·크기를 바꿔 다시 풉니다.

**미완료 / 질문:**

**총 풀이 시간:**

**참고한 자료 / 힌트:**

원본 노트북에 작성한 설명과 코드를 함께 제출하면 문제 조건에 따라 채점합니다.
공개 확인을 통과해도 독립 풀이·다음날 재현은 별도로 확인합니다.
